In [102]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.optimize import curve_fit

In [ ]:
df = pd.read_csv("../benchmarks/output/performance_detailed.csv")

def parse_reduced_size(s):
    try:
        r, c = map(int, s.split('×'))
        return pd.Series({'rows': r, 'cols': c, 'size': r * c})
    except:
        return pd.Series({'rows': 0, 'cols': 0, 'size': 0})

df = pd.concat([df, df['reduced_size'].apply(parse_reduced_size)], axis=1)
df = df.drop(columns=['reduced_size']+['file'])

df['sol_found'] = df['mhs_count'] > 0

df_comparison = df.copy()
df_comparison.drop(columns=['matrix_size', 'mhs_count', 'timeout', 'size_limit'])
df_comparison

fig = px.scatter_matrix(df_comparison,
    dimensions=["rows", "cols", "size", "ones_count","computation_time", "hypotheses_generated", "levels_explored", "max_level_size", "file_size_mb"],
    color="sol_found",
    symbol='sol_found',
    height = 1600)
fig.update_traces(diagonal_visible=False) # non ha senso mostrare una variabile rispetto a sè stessa
fig.show()

* Analisi della correlazione fra i vari dati relativi alle istanze risolte.
* Comparazione fra istanze che hanno trovato soluzione (blue) e quelle che non l'hanno raggiunta (red) -> ovvio notare che all'aumentare delle dimensioni del problema, l'algoritmo fa più fatica nella richerca, infatti è esponenziale, alcune curve particolari che bisognerebbe approfondire con tempi di risoluzione maggiori oppure maggiori dati
* cercar di capire come la complessità dell'istanza data in input influenza il possibile raggiungimento di una soluzione -> fare delle considerazioni (possibilmente non banali...)

In [104]:
df_failed = df.copy()
indices_to_drop = df_failed[df_failed['timeout'] != True].index
df_failed = df_failed.drop(indices_to_drop)

indices_to_drop = df[df['timeout']].index
df = df.drop(indices_to_drop)

to_drop_cols = [col for col in df.columns if not df[col].any() and df[col].dtype == 'bool'] # all columns that are empty and boolean
df = df.drop(columns=['matrix_size'] + to_drop_cols)
df_failed = df_failed.drop(columns=['matrix_size'] + to_drop_cols)

fig = px.parallel_coordinates(df, color="levels_explored", labels={"hypotheses_generated": "Ipotesi",
                  "levels_explored": "Livelli esplorati", "max_level_size": "Massima ampiezza",
                  "max_level_size": "Dimensione file", "ones_count": "numeri di uno", },
                    color_continuous_midpoint=4)
fig.show()


* analisi del dataset delle sole istanze che hanno trovato una soluzione -> maggior raccolta di dati analizzabili
* su 350 problemi proposti solo 98 hanno raggiunto soluzione in meno di un minuto -> aumentare tempi computazionali, considerare la complessità polinomiale dell'algoritmo, scremare le istanza più o meno complesse, fare run completo con timeout basso e campionare istanze complesse

In [105]:
fig = px.scatter(
    df,                 
    x='size',                      
    y='computation_time',                 
    color='max_level_size',
    # size='levels_explored',                 
    hover_data=['rows', 'levels_explored'],
    title='Distribuzione dei tempi di computazione in funzione della dimensione della matrice e dei livelli esplorati sulle istanze risolte',
    labels={'levels_explored': 'Livelli esplorati', 'computation_time': 'Tempo di computazione (s)', 'size': 'Dimensione della matrice', 'max_level_size': 'Massima ampiezza'},
    marginal_x="violin",
    marginal_y="violin",
    height=1100,
    width=1600
)

fig.show()

* analisi della distribuzione delle istanze risolte a seconda del tempo di esecuzione impiegato e della dimensione della matrice -> fino a dimensione 70 tempi computazionali molto bassi
* alcuni possibili outliers da dimensione 90 in poi, ma ciò potrebbe dipendere da fattori come
    1. numeri non nulli presenti nella matrice -> anche se si fa già dell'ottimizzazione con eliminazione di colonne vuote inutili al raggiungimento della soluzioni, queste matrici possono tendere ad essere molto sparse, il che le rende molto inefficienti se rappresentate nella loro interezza, perchè occupano molto spazio e da elaborare rischiano di essere più lunghe del necessario
    2. struttura della matrice stessa -> dove sono gli uno   
* vedere come all'aumentare delle dimensioni aumenta il tempo computazionale
* non sembra esserci correlazione fra massima ampiezza e livelli esplorati, come è ovvio pensare matrici piccole vengono risolte velocemente facendo poca esplorazione (=> pochi livelli esplorati con ampiezze minori)

In [106]:
df_failed.sort_values(by='size', ascending=True, inplace=True)
df_failed

fig = px.scatter(
    df_failed,                 
    x='size',                      
    y='computation_time',                 
    color='max_level_size',
    # size='levels_explored',                 
    hover_data=['rows', 'levels_explored'],
    title='Distribuzione dei tempi di computazione in funzione della dimensione della matrice e dei livelli esplorati sulle istanze non risolte',
    labels={'levels_explored': 'Livelli esplorati', 'computation_time': 'Tempo di computazione (s)', 'size': 'Dimensione della matrice', 'max_level_size': 'Massima ampiezza'},
    marginal_x="violin",
    marginal_y="violin",
    height=1100,
    width=1600
)

fig.show()

* analisi della distribuzione delle istanze non risolte a seconda del tempo di esecuzione impiegato e della dimensione della matrice -> ricontrollare la questione del tempo di esecuzione perchè ha poco senso che sia così variabile (le istanze dovrebbero arrivare tutte a timeout che è uguale per tutte - a meno che non venga già setacciato tutto lo spazio delle soluzioni, cosa che in questo caso è impossibile perchè il timeout era 60 e hanno tutte valori maggiori)
* vedere come all'aumentare delle dimensioni aumenta il tempo computazionale e diminuiscono i livelli esplorati perchè la generazione delle ipotesi diventa complessa
* notare che la massima ampiezza è minima quando si hanno molti livelli esplorati (cosa che accade quando le istanze sono meno complesse, quindi il grafo dello spazio di esplorazione è più ridotto) oppure quando si hanno pochissimi livelli esplorati (a volte uno solo) perchè l'istanza ha una matrice molto grande che non riesce ad elaborare abbastanza velocemente per esplorare sufficientemente lo spazio delle soluzioni.

In [107]:
# Definizione di funzioni utili per il corretto display dei dataset per i grafici

'''Pulisce il dataset mantenendo solo le feature a dati numerici.'''
def clean_dataset(df):
    df_numeric = df.copy()
    for col in df_numeric.columns:
        original_dtype = df_numeric[col].dtype
        df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')
        if df_numeric[col].isnull().all() and not pd.api.types.is_numeric_dtype(original_dtype):
            print(f"Avviso: La colonna '{col}' è diventata tutta NaN dopo la conversione. Probabilmente non era numerica.")

    initial_cols = set(df_numeric.columns)
    df_numeric.dropna(axis=1, how='all', inplace=True)
    dropped_cols_all_nan = list(initial_cols - set(df_numeric.columns))
    if dropped_cols_all_nan:
        print(f"\nColonne rimosse perché interamente NaN dopo conversione: {dropped_cols_all_nan}")
    else:
        print("\nNessuna colonna rimossa perché interamente NaN dopo conversione.")
    return df_numeric

'''Estrae una stringa con le statisiche da visualizzare per le distribuzioni dei dati.  
   Media, deviazione standard, mediana, minimo e massimo.'''
def extract_stats(col, plot_data):
    mean_val = plot_data[col].mean()
    std_val = plot_data[col].std()
    median_val = plot_data[col].median()
    min_val = plot_data[col].min()
    max_val = plot_data[col].max()

    stats_text = (
        f'µ={mean_val:.2f}<br>'
        f'σ={std_val:.2f}<br>'
        f'median={median_val:.2f}<br>'
        f'min={min_val:.2f}<br>'
        f'max={max_val:.2f}'
    )
    return stats_text

'''Aggiunge nota al grafico con dati aggiuntivi delle statistiche.
    Formattazione grafica con posizione a lato dx del grafico.'''
def add_annotation(stats_text, fig):
    fig.add_annotation(
        xref="paper",
        yref="paper",
        x=1.05,
        y=0.5,
        text=stats_text,
        showarrow=False,
        font=dict(
            size=12,
            color="black"
        ),
        align="left",
        xanchor='left',
        yanchor='middle',
        bgcolor="lightgrey",
        bordercolor="black",
        borderwidth=1,
        borderpad=5
        )

In [108]:
df_numeric = clean_dataset(df)
df_numeric = df_numeric.drop(['sol_found'], axis=1) # drop booleano flag usato per le precedenti matrici di correlazione fra set istanze risolte e non

cols_to_plot = [col for col in df_numeric.columns if pd.api.types.is_numeric_dtype(df_numeric[col])]

if not cols_to_plot:
    print("\nErrore: Nessuna colonna numerica valida trovata per il plotting.")
    exit()

for col in cols_to_plot:
    plot_data = df_numeric[[col]].dropna()

    try:
        stats_text = extract_stats(col, plot_data)

        fig = px.histogram(
            plot_data,
            x=col,
            histnorm='density',
            color_discrete_sequence=['red'],
            opacity=0.75,
            title=f'{col}',
            labels={col: col.replace("_", " ").title(), 'density': 'Density'},
            width=900, height=500,
            nbins=20, 
            marginal="violin"
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')

        add_annotation(stats_text, fig)

        fig.update_layout(
            margin=dict(
                l=80,  
                r=350, 
                b=80, 
                t=80   
            ),
        )

        fig.show()

    except Exception as e:
        print(f"  ERRORE: Non è stato possibile generare il grafico per '{col}'. Errore: {e}") # gestione di eventuali errori dovuti a non corretta conversione/pulizia dei dati


Nessuna colonna rimossa perché interamente NaN dopo conversione.


In [109]:
df_failed_numeric = clean_dataset(df_failed)
df_failed_numeric = df_failed_numeric.drop(columns=['mhs_count','sol_found'])
cols_to_plot = [col for col in df_failed_numeric.columns if pd.api.types.is_numeric_dtype(df_failed_numeric[col])]

if not cols_to_plot:
    print("\nErrore: Nessuna colonna numerica valida trovata per il plotting.")
    exit()

for col in cols_to_plot:
    plot_data = df_failed_numeric[[col]].dropna()

    try:
        stats_text = extract_stats(col, plot_data)

        fig = px.histogram(
            plot_data,
            x=col,
            histnorm='density',
            color_discrete_sequence=['blue'],
            opacity=0.75,
            title=f'{col}',
            labels={col: col.replace("_", " ").title(), 'density': 'Density'},
            width=900, height=500,
            nbins=20, 
            marginal="violin"
        )
        

        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')

        add_annotation(stats_text, fig)
        
        fig.update_layout(
            margin=dict(
                l=80,  
                r=350, 
                b=80, 
                t=80   
            ),
        )

        fig.show()

    except Exception as e:
        print(f"  ERRORE CRITICO: Non è stato possibile generare il grafico per '{col}'. Errore: {e}")


Nessuna colonna rimossa perché interamente NaN dopo conversione.


In [110]:
correlation_matrix = df.corr()

fig = px.imshow(correlation_matrix,
                text_auto=True, 
                # color_continuous_scale=px.colors.sequential.RdBu,
                # range_color=[-1, 1],
                title='Matrice di Correlazione tra Variabili',
                labels=dict(x="Variabile 1", y="Variabile 2", color="Correlazione"),
                width=1600, 
                height=1600 
               )

fig.update_xaxes(side="top") 
fig.update_layout(
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    xaxis_zeroline=False,
    yaxis_zeroline=False,
    coloraxis_colorbar=dict(
        title="Correlazione", 
        tickvals=[-1, 0, 1], 
        ticktext=["-1 (Negativa Forte)", "0 (Nessuna)", "1 (Positiva Forte)"]
    )
)

fig.show()


I valori che hanno un maggior indice di correlazione sono

*   `rows` e `level_explored` -> indice pari a 1 (grado massimo di correlazione)
*   `max_level_size` e `computational_time` -> indice pari a 0.87
*   `cols` e `mhs_count` -> indice pari a 0.85

Infatti si può intuitivamente comprendere che maggiore è il tempo coputazionale dedicato alla ricerca e risulta più probabile riuscire ad espandere la dimensione dell'ultimo livello esplorato.

Protrebbe quindi aver senso vedere quale sai l'andamento fra le due features correlate attraverso uno scatter.

In [111]:
fig = px.scatter(
    df.sort_values(by='rows'),                 
    x='rows',                      
    y='levels_explored',                  
    color='levels_explored',
    size='levels_explored',                 
    hover_data=['rows', 'levels_explored'],
    title='Scatter Plot Interattivo: levels_explored vs rows',
    labels={'rows': 'Righe della matrice', 'levels_explored': 'Livelli esplorati'},
    marginal_x="box"
)

fig.show()

In [112]:
df['mhs_cat'] = pd.cut(df['mhs_count'], bins=5, labels=['1','2','3','4','5'], right=True, include_lowest=True)
fig = px.scatter_matrix(df, height=1600, width=1600, color="mhs_cat")
fig.show()

In [113]:
fig = px.scatter(
    df,                 
    x='size',                      
    y='computation_time',                  
    color='mhs_count',
    size='mhs_count',                 
    hover_data=['size', 'computation_time', 'mhs_count'],
    title='Scatter Plot Interattivo: computation_time vs size',
    labels={'size': 'Elementi totali della matrice', 'computation_time': 'Tempo computazionale (s)', 'mhs_count': 'Numero di MHS'}
    # marginal_x="box"
)

fig.show()

In [114]:
fig = px.scatter(
    df,
    x='size',
    y='computation_time',
    color='ones_count',  # Colora i punti in base al numero di '1'
    size='ones_count', # La dimensione del punto indica il numero di '1' nella matrice
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb', 'ones_count', 'cols'],
    title='Tempo di Calcolo vs. Dimensione della Matrice (Colorato per 1)',
    labels={'size': 'Dimensione Matrice (righe*colonne)', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True # Utile se il tempo di calcolo cresce esponenzialmente
)
fig.show()

In [115]:
fig = px.scatter(
    df,
    x='hypotheses_generated',
    y='computation_time',
    color='size', # Colora in base alla dimensione del problema
    size='size', # La dimensione del punto indica il numero di MHS trovati
    # hover_data=['mhs_count', 'levels_explored', 'max_level_size', 'rows', 'cols', 'file_size_mb'],
    # title='Tempo di Calcolo vs. Ipotesi Generate (Colorato per Dimensione Matrice)',
    # labels={'hypotheses_generated': 'Ipotesi Generate', 'computation_time': 'Tempo di Calcolo (s)'},
    log_x=True, # Utile se le ipotesi generate crescono molto
    log_y=True
)
fig.show()

In [116]:
df['cols_binned'] = pd.cut(df['cols'], bins=6)
df['rows_binned'] = pd.cut(df['rows'], bins=3)

In [117]:
fig = px.box(
    df,
    x='rows',
    y='computation_time',
    color='cols_binned', # Mostra box plot separati per 'rows', colorati per 'cols'
    points="all", # Mostra tutti i punti dati oltre al box plot
    hover_data=['mhs_count', 'size', 'file_size_mb', 'ones_count'],
    title='Distribuzione del Tempo di Calcolo per Numero di Righe e Colonne',
    labels={'rows': 'Numero di Righe', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True
)
fig.show()

In [118]:
fig = px.scatter(
    df,
    x='size',
    y='computation_time',
    color='ones_count', # Colora in base al numero di '1'
    facet_col='cols_binned', # Crea colonne separate per ogni valore di 'cols'
    facet_row='rows_binned', # Crea righe separate per ogni valore di 'rows'
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb'],
    title='Tempo di Calcolo vs. Dimensione Matrice per Righe e Colonne',
    labels={'size': 'Dimensione Matrice', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True,
    height=800 # Aumenta l'altezza per una migliore visualizzazione dei facet
)
fig.show()

In [119]:
df_sorted = df.sort_values(by=['rows', 'cols', 'size'])

fig = px.line(
    df_sorted,
    x='size',                  # Variabile sull'asse X (dimensione della matrice)
    y='computation_time',      # Variabile sull'asse Y (tempo di calcolo)
    color='cols_binned',              # Crea una linea separata e colorata per ogni valore di 'rows'
    line_dash='rows_binned',          # (Opzionale) Aggiunge uno stile di linea diverso per ogni valore di 'cols'
    hover_data=['mhs_count', 'ones_count', 'rows', 'cols'],
    title='Tempo di Calcolo vs. Dimensione Matrice per Diverse Config. (rows/cols)',
    labels={'size': 'Dimensione Matrice (righe*colonne)', 'computation_time': 'Tempo di Calcolo (s)',
            'rows_binned': 'Num. Righe', 'cols_binned': 'Num. Colonne'},
    log_y=True,                # Utile se il tempo di calcolo cresce esponenzialmente
    markers=True               # Mostra un marcatore per ogni punto dato
)

fig.update_traces(mode='lines+markers') # Assicurati che vengano mostrati sia linee che marcatori

fig.show()

In [120]:
def compare_function(fig, x_data, y_data):
    # --- Regressione Lineare ---
    def linear_func(x, a, b):
        return a * x + b
    try:
        popt_linear, _ = curve_fit(linear_func, x_data, y_data)
        y_fit_linear = linear_func(x_data, *popt_linear)
        fig.add_scatter(x=x_data, y=y_fit_linear, mode='lines',
                    name=f'Linear (y={popt_linear[0]:.2f}x + {popt_linear[1]:.2f})',
                    line=dict(dash='dash', color='blue'))
    except RuntimeError:
        pass

# --- Regressione Esponenziale ---
    def exponential_func(x, a, b, c):
        return a * np.exp(b * x) + c
    try:
    # Aggiornati i parametri iniziali per riflettere un esponente atteso più grande
        popt_exp, _ = curve_fit(exponential_func, x_data, y_data, p0=[1, 0.05, 0], maxfev=5000)
        y_fit_exp = exponential_func(x_data, *popt_exp)
        fig.add_scatter(x=x_data, y=y_fit_exp, mode='lines',
                    name=f'Exponential (y={popt_exp[0]:.2f}e^({popt_exp[1]:.2f}x) + {popt_exp[2]:.2f})',
                    line=dict(dash='dashdot', color='green'))
    except RuntimeError:
        pass

# --- Regressione Polinomiale (Grado 2) ---
    degree = 2
    try:
        coefficients_poly = np.polyfit(x_data, y_data, degree)
        y_fit_poly = np.polyval(coefficients_poly, x_data)
        fig.add_scatter(x=x_data, y=y_fit_poly, mode='lines',
                    name=f'Polynomial (Deg {degree})',
                    line=dict(dash='dot', color='yellow'))
    except Exception:
        pass

    degree = 3
    try:
        coefficients_poly = np.polyfit(x_data, y_data, degree)
        y_fit_poly = np.polyval(coefficients_poly, x_data)
        fig.add_scatter(x=x_data, y=y_fit_poly, mode='lines',
                    name=f'Polynomial (Deg {degree})',
                    line=dict(dash='dot', color='orange'))
    except Exception:
        pass

    degree = 4
    try:
        coefficients_poly = np.polyfit(x_data, y_data, degree)
        y_fit_poly = np.polyval(coefficients_poly, x_data)
        fig.add_scatter(x=x_data, y=y_fit_poly, mode='lines',
                    name=f'Polynomial (Deg {degree})',
                    line=dict(dash='dot', color='red'))
    except Exception:
        pass


In [121]:
df_sorted = df.sort_values('size')
fig = px.line(
    df_sorted,
    x='size',
    y='computation_time',
    title='Computation Time vs. Number of Elements',
    labels={'size': 'Number of Elements', 'computation_time': 'Computation Time (s)'},
    markers=True
)
x_data = df_sorted['size']
y_data = df_sorted['computation_time']

compare_function(fig, x_data, y_data)
fig.update_layout(showlegend=True)
fig.show()


In [122]:
df_sorted = df.sort_values('mhs_count')
fig = px.line(
    df_sorted,
    x='mhs_count',
    y='computation_time',
    title='Computation Time vs. Number of MHS found',
    labels={'mhs_count': 'Number of MHS found', 'computation_time': 'Computation Time (s)'},
    markers=True
)

x_data = df_sorted['mhs_count']
y_data = df_sorted['computation_time']

compare_function(fig, x_data, y_data)
fig.update_layout(showlegend=True)
fig.show()


In [123]:
df_sorted = df.sort_values('mhs_count')

fig = px.line(
    df_sorted,
    x='mhs_count',
    y='size',
    title='Size vs. Number of MHS found with Regression Curves',
    labels={'mhs_count': 'Number of MHS found', 'size': 'Size'},
    markers=True
)

x_data = df_sorted['mhs_count']
y_data = df_sorted['size']

compare_function(fig, x_data, y_data)
fig.update_layout(showlegend=True)
fig.show()

In [124]:
fig = px.line(
    df.sort_values('file_size_mb'),
    x='file_size_mb',
    y='computation_time',
    title='Computation Time vs. File Size',
    labels={'file_size_mb': 'File Size (MB)', 'computation_time': 'Computation Time (s)'},
    markers=True,
    height=500,  # Altezza del grafico
    width=1600   # Larghezza del grafico
)

fig.show()

In [125]:
fig = px.line(
    df.sort_values('levels_explored'),
    x='levels_explored',
    y='computation_time',
    title='Computation Time vs. Levels Explored',
    labels={'levels_explored': 'Levels Explored', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [126]:
df_sorted = df.sort_values('max_level_size')
fig = px.line(
    df_sorted,
    x='max_level_size',
    y='computation_time',
    title='Computation Time vs. Max Level Size',
    labels={'max_level_size': 'Max Level Size', 'computation_time': 'Computation Time (s)'},
    markers=True 
)
x_data = df_sorted['max_level_size']
y_data = df_sorted['computation_time']

compare_function(fig, x_data, y_data)
fig.update_layout(showlegend=True)
fig.show()

/home/elena-margherita-nava/Documents/ElaboratoAlgoritmi/.plot_env/lib/python3.12/site-packages/scipy/optimize/_minpack_py.py:1024: RuntimeWarning:

overflow encountered in square

/home/elena-margherita-nava/Documents/ElaboratoAlgoritmi/.plot_env/lib/python3.12/site-packages/scipy/optimize/_minpack_py.py:1062: RuntimeWarning:

invalid value encountered in multiply

